# Lista 2 — Interpolação Polinomial

Notebook preparado para rodar no **Jupyter** com:
- enunciado reescrito;
- contexto das funções;
- resolução passo a passo;
- funções genéricas reutilizáveis.

Sempre que fizer sentido, eu mostro:
- os **nós usados**;
- o **polinômio/interpolador**;
- a **avaliação numérica** pedida;
- uma **comparação** com outra estratégia.

In [1]:
import numpy as np
import sympy as sp

np.set_printoptions(precision=8, suppress=True)
x = sp.symbols('x')

## Contexto das funções utilizadas

Funções genéricas usadas neste notebook:

- `lagrange_poly(xs, ys)`: monta o polinômio simbólico de Lagrange;
- `divided_differences(xs, ys)`: tabela de diferenças divididas;
- `newton_eval(xs, coeffs, xv)`: avalia o polinômio de Newton;
- `barycentric_eval(xs, ys, xv)`: avaliação numérica estável do interpolador;
- `linear_spline_eval(xs, ys, xv)`: spline linear;
- `quadratic_spline_eval(xs, ys, xv)`: spline quadrática simples com condição inicial padrão.

> Observação importante: para conjuntos com escalas muito grandes ou grau muito alto, a **avaliação baricêntrica** costuma ser mais estável do que expandir o polinômio.

In [2]:
def lagrange_poly(xs, ys, symbol=x):
    xs = list(map(sp.nsimplify, xs))
    ys = list(map(sp.nsimplify, ys))
    poly = 0
    for i, xi in enumerate(xs):
        Li = 1
        for j, xj in enumerate(xs):
            if i != j:
                Li *= (symbol - xj)/(xi - xj)
        poly += ys[i] * Li
    return sp.expand(poly)

def divided_differences(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    table = np.zeros((n, n), dtype=float)
    table[:, 0] = ys
    for j in range(1, n):
        for i in range(n - j):
            table[i, j] = (table[i+1, j-1] - table[i, j-1]) / (xs[i+j] - xs[i])
    return table[0, :], table

def newton_eval(xs, coeffs, xv):
    xs = np.asarray(xs, dtype=float)
    coeffs = np.asarray(coeffs, dtype=float)
    xv = np.asarray(xv, dtype=float)
    result = np.zeros_like(xv, dtype=float) + coeffs[-1]
    for k in range(len(coeffs) - 2, -1, -1):
        result = result * (xv - xs[k]) + coeffs[k]
    return result

def newton_poly_symbolic(xs, ys, symbol=x):
    coeffs, _ = divided_differences(xs, ys)
    expr = sp.Float(coeffs[0])
    prod = 1
    for k in range(1, len(coeffs)):
        prod *= (symbol - sp.nsimplify(xs[k-1]))
        expr += sp.Float(coeffs[k]) * prod
    return sp.expand(expr)

def barycentric_weights(xs):
    xs = np.asarray(xs, dtype=float)
    n = len(xs)
    w = np.ones(n, dtype=float)
    for j in range(n):
        for k in range(n):
            if j != k:
                w[j] /= (xs[j] - xs[k])
    return w

def barycentric_eval(xs, ys, xv):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    xv = np.atleast_1d(np.asarray(xv, dtype=float))
    w = barycentric_weights(xs)
    out = []
    for x0 in xv:
        diff = x0 - xs
        idx = np.where(np.isclose(diff, 0))[0]
        if len(idx) > 0:
            out.append(ys[idx[0]])
        else:
            terms = w / diff
            out.append(np.sum(terms * ys) / np.sum(terms))
    return np.array(out)

def linear_spline_eval(xs, ys, xv):
    return np.interp(xv, xs, ys)

def quadratic_spline_coeffs(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs) - 1
    h = np.diff(xs)

    a = ys[:-1].copy()
    b = np.zeros(n, dtype=float)
    c = np.zeros(n, dtype=float)

    # condição extra: c0 = 0
    c[0] = 0.0
    b[0] = (ys[1] - ys[0]) / h[0]

    for i in range(1, n):
        b[i] = b[i-1] + 2*c[i-1]*h[i-1]
        c[i] = (ys[i+1] - ys[i] - b[i]*h[i]) / (h[i]**2)

    return a, b, c

def quadratic_spline_eval(xs, ys, xv):
    xs = np.asarray(xs, dtype=float)
    xv = np.atleast_1d(np.asarray(xv, dtype=float))
    a, b, c = quadratic_spline_coeffs(xs, ys)
    out = np.zeros_like(xv, dtype=float)
    for ind, x0 in np.ndenumerate(xv):
        i = np.searchsorted(xs, x0) - 1
        i = max(0, min(i, len(xs) - 2))
        dx = x0 - xs[i]
        out[ind] = a[i] + b[i]*dx + c[i]*dx**2
    return out

def max_abs_error(y_true, y_pred):
    return float(np.max(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

## Questão 1

**Enunciado reescrito.** Gerar 5 pontos igualmente espaçados de tensão entre 0,1 V e 0,7 V para o diodo
\[
I = I_s\left(e^{\frac{V}{\nu V_T}} - 1\right),
\]
depois construir o interpolador de Lagrange nos pontos \((I, V)\) e comparar com a função exata em pontos intermediários.

**Observação.** Os parâmetros do enunciado geram correntes muito grandes. Isso deixa a interpolação polinomial numericamente sensível — justamente o tipo de situação em que vale a pena comparar o interpolador com a função exata.

In [3]:
Is = 10.0
nu = 1.8
VT = 26e-3

V_nodes = np.linspace(0.1, 0.7, 5)
I_nodes = Is * (np.exp(V_nodes/(nu*VT)) - 1.0)

print("Nós em V:")
print(V_nodes)
print("\nNós em I:")
print(I_nodes)

# polinômio simbólico de Lagrange em V(I)
P_VI = lagrange_poly(I_nodes, V_nodes, symbol=x)

# comparação em pontos intermediários entre os nós de V
V_mid = 0.5*(V_nodes[:-1] + V_nodes[1:])
I_mid = Is * (np.exp(V_mid/(nu*VT)) - 1.0)
V_interp = barycentric_eval(I_nodes, V_nodes, I_mid)

print("\nComparação em pontos intermediários:")
for vv, ii, vi in zip(V_mid, I_mid, V_interp):
    print(f"V exato = {vv:.6f} V | I correspondente = {ii:.6e} A | V interpolado = {vi:.6f} V | erro = {vi - vv:.6f}")

print("\nComentário:")
print("- O interpolador existe, mas o problema fica mal condicionado por causa da escala enorme de I.")
print("- Isso ajuda a explicar os erros elevados em alguns pontos intermediários.")

Nós em V:
[0.1  0.25 0.4  0.55 0.7 ]

Nós em I:
[      74.71877398     2079.0515432     51503.21419256  1270236.89509973
 31322577.8020131 ]

Comparação em pontos intermediários:
V exato = 0.175000 V | I correspondente = 4.106922e+02 A | V interpolado = 0.125959 V | erro = -0.049041
V exato = 0.325000 V | I correspondente = 1.036371e+04 A | V interpolado = 0.746992 V | erro = 0.421992
V exato = 0.475000 V | I correspondente = 2.557917e+05 A | V interpolado = -56.328369 V | erro = -56.803369
V exato = 0.625000 V | I correspondente = 6.307717e+06 A | V interpolado = 174096.325990 V | erro = 174095.700990

Comentário:
- O interpolador existe, mas o problema fica mal condicionado por causa da escala enorme de I.
- Isso ajuda a explicar os erros elevados em alguns pontos intermediários.


## Questão 2

**Enunciado reescrito.** Para o termistor
\[
R(T) = R_0 e^{\beta\left(\frac{1}{T} - \frac{1}{T_0}\right)},
\]
usar 7 pontos entre 20°C e 100°C e construir um polinômio \(T(R)\). Depois avaliar a precisão em pontos intermediários.

Como a fórmula original usa temperatura absoluta, a conversão adotada será:
\[
T_{K} = T_{°C} + 273{,}15.
\]

In [4]:
R0 = 10000.0
T0 = 298.0
beta = 4000.0

Tc_nodes = np.linspace(20, 100, 7)
Tk_nodes = Tc_nodes + 273.15
R_nodes = R0 * np.exp(beta*(1/Tk_nodes - 1/T0))

print("Temperaturas (°C):")
print(Tc_nodes)
print("\nResistências correspondentes (ohms):")
print(R_nodes)

# interpolador numérico T(R)
Tc_mid = 0.5*(Tc_nodes[:-1] + Tc_nodes[1:])
Tk_mid = Tc_mid + 273.15
R_mid = R0 * np.exp(beta*(1/Tk_mid - 1/T0))
Tc_interp = barycentric_eval(R_nodes, Tc_nodes, R_mid)

print("\nComparação em pontos intermediários:")
for t_true, r_val, t_hat in zip(Tc_mid, R_mid, Tc_interp):
    print(f"T exata = {t_true:8.4f} °C | R = {r_val:10.4f} ohms | T interpolada = {t_hat:10.4f} °C | erro = {t_hat - t_true:10.4f}")

print(f"\nErro absoluto máximo nos pontos intermediários: {max_abs_error(Tc_mid, Tc_interp):.6f} °C")

print("\nComentário:")
print("- O polinômio fornece uma aproximação local, mas há forte não linearidade entre T e R.")
print("- Nos extremos do intervalo a oscilação do polinômio pode crescer.")

Temperaturas (°C):
[ 20.          33.33333333  46.66666667  60.          73.33333333
  86.66666667 100.        ]

Resistências correspondentes (ohms):
[12486.62404191  6896.72704637  4002.55082712  2426.30599254
  1528.57073593   996.54471474   669.86175746]

Comparação em pontos intermediários:
T exata =  26.6667 °C | R =  9218.8727 ohms | T interpolada =  -214.0892 °C | erro =  -240.7558
T exata =  40.0000 °C | R =  5223.6561 ohms | T interpolada =    48.4462 °C | erro =     8.4462
T exata =  53.3333 °C | R =  3100.4283 ohms | T interpolada =    52.5194 °C | erro =    -0.8140
T exata =  66.6667 °C | R =  1917.1109 ohms | T interpolada =    66.8475 °C | erro =     0.1808
T exata =  80.0000 °C | R =  1229.2424 ohms | T interpolada =    79.9114 °C | erro =    -0.0886
T exata =  93.3333 °C | R =   814.0891 ohms | T interpolada =    93.4378 °C | erro =     0.1044

Erro absoluto máximo nos pontos intermediários: 240.755827 °C

Comentário:
- O polinômio fornece uma aproximação local, mas há

## Questão 3

**Enunciado reescrito.** Ajustar um polinômio interpolador aos dados de carga horária, prever a carga às 4h30 e comparar com spline linear.

In [5]:
hora = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
mw = np.array([45, 42, 40, 38, 36, 40, 50, 65], dtype=float)

P = lagrange_poly(hora, mw, symbol=x)
pred_poly = barycentric_eval(hora, mw, [4.5])[0]
pred_spline = linear_spline_eval(hora, mw, [4.5])[0]

print("Polinômio interpolador (expandido):")
display(sp.expand(P))

print(f"Carga prevista pelo polinômio em 4.5 h = {pred_poly:.6f} MW")
print(f"Carga prevista pela spline linear em 4.5 h = {pred_spline:.6f} MW")
print(f"Diferença entre as duas estratégias = {pred_poly - pred_spline:.6f} MW")

Polinômio interpolador (expandido):


17*x**7/2520 - 17*x**6/80 + 1951*x**5/720 - 287*x**4/16 + 47503*x**3/720 - 2667*x**2/20 + 18593*x/140 - 5

Carga prevista pelo polinômio em 4.5 h = 36.625000 MW
Carga prevista pela spline linear em 4.5 h = 37.000000 MW
Diferença entre as duas estratégias = -0.375000 MW


## Questão 4

**Enunciado reescrito.** Para os dados
\[
(1,5), (2,2), (5,4), (7,8), (9,3),
\]
calcular o valor em \(x=4{,}8\) usando:
- Newton de 2º grau;
- Lagrange de 2º grau;
- spline quadrática.

**Escolha dos três pontos para o polinômio de 2º grau.**  
Como \(x=4{,}8\) está perto de 5 e entre 2 e 7, a escolha mais natural é usar os pontos:
\[
x = 2, 5, 7.
\]

In [6]:
xs = np.array([1, 2, 5, 7, 9], dtype=float)
ys = np.array([5, 2, 4, 8, 3], dtype=float)
xq = 4.8

x_local = np.array([2, 5, 7], dtype=float)
y_local = np.array([2, 4, 8], dtype=float)

coef_newton, tabela = divided_differences(x_local, y_local)
val_newton = float(newton_eval(x_local, coef_newton, [xq])[0])
P_lagrange = lagrange_poly(x_local, y_local, symbol=x)
val_lagrange = float(P_lagrange.subs(x, xq))
val_spline = float(quadratic_spline_eval(xs, ys, [xq])[0])

print("Polinômio de Newton (expandido):")
display(newton_poly_symbolic(x_local, y_local))

print("\nPolinômio de Lagrange (expandido):")
display(P_lagrange)

print(f"\nValor em x = {xq}")
print(f"Newton (2º grau)      = {val_newton:.6f}")
print(f"Lagrange (2º grau)    = {val_lagrange:.6f}")
print(f"Spline quadrática     = {val_spline:.6f}")

Polinômio de Newton (expandido):


0.266666666666667*x**2 - 1.2*x + 3.33333333333333


Polinômio de Lagrange (expandido):


4*x**2/15 - 6*x/5 + 10/3


Valor em x = 4.8
Newton (2º grau)      = 3.717333
Lagrange (2º grau)    = 3.717333
Spline quadrática     = 3.182222


## Questão 5

**Enunciado reescrito.** Estimar \(f(2{,}8)\) usando polinômios de Newton de 1º, 2º e 3º graus, escolhendo a sequência de pontos para obter a melhor acurácia possível.

**Critério adotado.** Escolher os nós mais próximos de 2,8:
- grau 1: \(2{,}5, 3{,}2\);
- grau 2: \(2, 2{,}5, 3{,}2\);
- grau 3: \(2, 2{,}5, 3{,}2, 4\).

In [7]:
xs = np.array([1.6, 2.0, 2.5, 3.2, 4.0, 4.5], dtype=float)
fx = np.array([2, 8, 14, 15, 8, 2], dtype=float)
xq = 2.8

escolhas = {
    "grau_1": np.array([2.5, 3.2], dtype=float),
    "grau_2": np.array([2.0, 2.5, 3.2], dtype=float),
    "grau_3": np.array([2.0, 2.5, 3.2, 4.0], dtype=float),
}

for nome, nos in escolhas.items():
    y_nos = np.array([fx[np.where(xs == xi)[0][0]] for xi in nos], dtype=float)
    coef, _ = divided_differences(nos, y_nos)
    valor = float(newton_eval(nos, coef, [xq])[0])
    print(f"{nome}: nós = {nos} -> f(2.8) ≈ {valor:.6f}")

grau_1: nós = [2.5 3.2] -> f(2.8) ≈ 14.428571
grau_2: nós = [2.  2.5 3.2] -> f(2.8) ≈ 15.485714
grau_3: nós = [2.  2.5 3.2 4. ] -> f(2.8) ≈ 15.388571


## Questão 6

**Enunciado reescrito.** Usar **interpolação inversa** para encontrar o valor de \(x\) que produz \(f(x)=0{,}85\), sabendo que a tabela foi gerada por
\[
f(x) = \frac{x^3}{2 + x^3}.
\]

Parte (a): obter o valor exato analiticamente.  
Parte (b): obter uma aproximação por interpolação cúbica de \(x\) em função de \(y\).

In [8]:
xs = np.array([0, 1, 2, 3, 4, 5], dtype=float)
ys = np.array([0, 0.5, 0.8, 0.9, 0.941176, 0.961538], dtype=float)
yq = 0.85

# valor exato
x_exact = (1.7/0.15)**(1/3)

# escolha de 4 pontos que cercam a região de interesse
idx = [1, 2, 3, 4]  # y = 0.5, 0.8, 0.9, 0.941176
y_local = ys[idx]
x_local = xs[idx]

coef_inv, _ = divided_differences(y_local, x_local)
x_interp = float(newton_eval(y_local, coef_inv, [yq])[0])

print(f"Valor analítico exato: x = {x_exact:.12f}")
print(f"Interpolação cúbica inversa: x ≈ {x_interp:.12f}")
print(f"Erro absoluto = {abs(x_interp - x_exact):.12f}")

Valor analítico exato: x = 2.246221366935
Interpolação cúbica inversa: x ≈ 2.290689697377
Erro absoluto = 0.044468330442


## Questão 7

**Enunciado reescrito.** No circuito com resistor linear \(R_1\) e varistor \(R_2\), usar interpolação com todos os pontos para estimar:
- a corrente \(I\) quando \(E_2 = 2{,}3\) V;
- a tensão \(E_1\), sabendo que \(R_1 = 10\ \Omega\);
- a tensão total \(E = E_1 + E_2\).

As relações fornecidas são:
\[
I = \frac{E_1}{R_1} = P_n(E_2),
\qquad
E = E_1 + E_2.
\]

In [9]:
E2 = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5], dtype=float)
I  = np.array([0.0, 0.0125, 0.06, 0.195, 0.5, 1.0875, 2.1, 3.71, 6.12, 9.5625], dtype=float)

E2_q = 2.3
R1 = 10.0

I_q = float(barycentric_eval(E2, I, [E2_q])[0])
E1_q = R1 * I_q
E_total = E1_q + E2_q

print(f"I(2.3 V) ≈ {I_q:.6f} A")
print(f"E1 ≈ {E1_q:.6f} V")
print(f"E  ≈ {E_total:.6f} V")

I(2.3 V) ≈ 0.810152 A
E1 ≈ 8.101520 V
E  ≈ 10.401520 V
